# Descrizione del CSV generato da `bench_e2e_alt.rs`

## Panoramica

Il file è un **CSV con header**, una riga per ogni run del benchmark. Viene creato in `bench_alg2.csv` (o nel path dato con `--out`), e con `--append` le run di invocazioni successive si accumulano nello stesso file.

```
run,n,u,n_bits,hash_k,degree,set_size,overlap,backend,total_s,sigma,expected_intersection_size,found_intersection_size,false_positives,false_negatives,match
0,2,4,128,3,8192,6,2,bfv+pasta3-alt,1.234567,2,2,2,0,0,true
1,2,4,128,3,8192,6,2,bfv+pasta3-alt,1.198432,1,2,2,0,0,true
2,2,4,128,3,8192,6,2,bfv+pasta3-alt,1.211029,0,2,2,0,0,true
```

**16 colonne**, tutte separate da virgola. Le prime 8 sono **parametri di quella run** (identici per tutte le run di una singola invocazione, diversi se usi `--append` con parametri cambiati). Le altre 8 sono **misure/output** di quella run.

---

## Colonna per colonna

| # | Nome | Tipo | Significato | Esempio | Come usarlo nei grafici |
|---|------|------|-------------|---------|--------------------------|
| 1 | `run` | int | Indice della run **all'interno dell'invocazione** (0-based). Con `--append` **si resetta** a 0 per ogni invocazione: per un identificatore unico serve una colonna tua (vedi sotto). | `0`, `1`, `2` | Solo per debug, non plottare. |
| 2 | `n` | int | Numero di Data Owner (`--n`). | `2`, `3`, `5` | **Asse X** tipico per scalare il numero di partecipanti. |
| 3 | `u` | int | Numero di hash family / scelte OT (`--u`). | `2`, `4`, `8` | Asse X per studiare l'effetto della plausible deniability. |
| 4 | `n_bits` | int | Lunghezza Bloom filter in bit (`--n-bits`). Deve essere ≤ `PASTA3.t = 128`. | `32`, `64`, `128` | Asse X per l'effetto della dimensione del filtro su FP. |
| 5 | `hash_k` | int | Numero di funzioni hash per family (`--hash-k`). | `2`, `3`, `4` | Asse X per trade-off FP vs costo. |
| 6 | `degree` | str/int | Grado FHE (`--degree`). Se passi `--degree 0` diventa `"secure"` (stringa); altrimenti è un intero. | `8192`, `16384`, `"secure"` | Colonna categorica (usala come `facet`/`color`). |
| 7 | `set_size` | int | Elementi per DO (`--set-size`). | `4`, `6`, `9` | **Asse X** principale per scalare la dimensione dei dataset. |
| 8 | `overlap` | int | Elementi condivisi tra i DO (`--overlap`). Default = `set_size/3`. | `0`, `2`, `3` | Usalo per calcolare la cardinalità attesa dell'intersezione (dovrebbe essere ≈ `overlap`, modulo FP). |
| 9 | `backend` | str | Identificatore fisso del protocollo: `"bfv+pasta3-alt"`. | `"bfv+pasta3-alt"` | Utile se in futuro accumuli anche Alg. 1 (`"bfv+pasta3"`) nello stesso CSV: **filtra su questa colonna** per separare i due protocolli. |
| 10 | `total_s` | float (6 dec.) | Tempo **wall-clock totale** di `run_e2e_pasta_alt`, in secondi. Include FHE setup, pack_key, cifratura SIM, OT, transcipher, AND, "CMUX" simulato, ground-truth. | `1.234567` | **Asse Y** principale (tempo). |
| 11 | `sigma` | int | Indice σ scelto dal TTP, in `[0, u)`. Varia ad ogni run. | `0`, `1`, `2`, `3` | Solo per debug / verificare che σ sia uniforme su più run. |
| 12 | `expected_intersection_size` | int | Cardinalità **verità** dell'intersezione `|∩ X^(a)|`. | `2` | Asse Y in un bar chart "expected vs found". |
| 13 | `found_intersection_size` | int | Cardinalità dell'intersezione **trovata** (token ≠ 0). | `2` | Confronta con la 12: devono coincidere. |
| 14 | `false_positives` | int | `|found \ expected|`. **Deve essere 0** per correttezza. | `0` | Se > 0: Bloom filter troppo piccolo (aumenta `n_bits` o `hash_k`). |
| 15 | `false_negatives` | int | `|expected \ found|`. **Deve essere 0** (no falsi negativi per costruzione del BF). Se > 0 è un bug. | `0` | Se > 0: guarda log `[psi-alt]` per capire dove fallisce. |
| 16 | `match` | bool | `"true"` se `FP=0 && FN=0`, `"false"` altrimenti. Colonna derivata, comoda per filtrare. | `"true"` | Filtra `match=true` prima di plottare i tempi (escludi run fallite). |

---

## Esempio di utilizzo per i grafici

**Scenario tipico**: sweep su `n` (2, 3, 5) con `--append`, poi plot `total_s` vs `n`.

```bash
# Invocazione 1
cargo run --release --example bench_e2e_alt -- \
  --n 2 --u 4 --n-bits 128 --hash-k 3 --set-size 6 --runs 5 \
  --out sweep_n.csv

# Invocazione 2
cargo run --release --example bench_e2e_alt -- \
  --n 3 --u 4 --n-bits 128 --hash-k 3 --set-size 6 --runs 5 \
  --out sweep_n.csv --append

# Invocazione 3
cargo run --release --example bench_e2e_alt -- \
  --n 5 --u 4 --n-bits 128 --hash-k 3 --set-size 6 --runs 5 \
  --out sweep_n.csv --append
```

Il CSV conterrà 15 righe (5 per ogni `n`). Il campo `run` sarà `0..4` tre volte (una per invocazione): **non è un identificatore unico**. Per distinguere le invocazioni ti servono `n` (cambia tra i blocchi) e l'ordine delle righe.

### In Python/pandas

```python
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("sweep_n.csv")
df = df[df["match"] == "true"]        # escludi run fallite
df = df[df["backend"] == "bfv+pasta3-alt"]  # se accumuli anche Alg. 1

# Tempo medio ± std per ogni n
agg = df.groupby("n")["total_s"].agg(["mean", "std"]).reset_index()
print(agg)

plt.errorbar(agg["n"], agg["mean"], yerr=agg["std"], marker="o", capsize=4)
plt.xlabel("n (Data Owners)")
plt.ylabel("total_s (s)")
plt.title("Alg. 2 — tempo totale vs n")
plt.grid(True)
plt.savefig("total_s_vs_n.png", dpi=150)
```

Per uno sweep multi-dimensionale (`n` × `u`), fai pivot:

```python
pivot = df.pivot_table(index="n", columns="u", values="total_s", aggfunc="mean")
pivot.plot(marker="o")
plt.xlabel("n"); plt.ylabel("total_s (s)")
plt.title("Alg. 2 — tempo totale vs n, una linea per u")
plt.grid(True)
plt.savefig("total_s_vs_n_by_u.png", dpi=150)
```

Per verificare la correttezza (i FP/FN devono essere sempre 0):

```python
assert (df["false_positives"] == 0).all()
assert (df["false_negatives"] == 0).all()
assert (df["match"] == "true").all()
```

---

## Note e avvertenze

### 1. `total_s` è **wall-clock** dell'intera `run_e2e_pasta_alt`
Include FHE setup (`PastaBfv::new_with_sop`), che domina il tempo. Se vuoi separare le fasi (setup / OT / transcipher / AND / CMUX) devi modificare `E2eResultAlt` come descritto nella risposta precedente (Opzione B): il file CSV attuale **non** le espone. Con `n_bits=128`, `hash_k=3`, `set_size=6`, `degree=8192`, la maggior parte di `total_s` è FHE setup; la parte "protocollo" (OT + transcipher + AND + CMUX) è dell'ordine dei secondi/minuti.

### 2. Precisione
`total_s` è stampato con `{:.6}` → risoluzione al microsecondo, ma il clock di sistema è più grossolano. Non attenderti valori significativi oltre il millisecondo.

### 3. Run fallite
Se `run_e2e_pasta_alt` restituisce `Err`, la riga diventa:
```
0,2,4,128,3,8192,6,2,bfv+pasta3-alt,FAILED,,,,,,,,
```
**Attenzione**: quella riga ha **17 campi** invece di 16 per un piccolo bug di `write_failed` (7 virgole dopo `FAILED` invece di 6). La maggior parte dei parser CSV lo tratta come colonna extra vuota, ma alcuni (es. `pd.read_csv` con `strict=True`) si lamentano. Con la validazione a priori del vincolo `set_size·hash_k·⌈log₂ n_bits⌉ ≤ 128` **non dovresti mai vedere FAILED**, quindi in pratica non è un problema. Se vuoi essere sicuro, filtra `df[df["total_s"] != "FAILED"]` **prima** di convertire `total_s` a float.

### 4. `run` non è unico tra invocazioni `--append`
Se vuoi un identificatore globale (es. per unire il CSV ad altri dati), aggiungi una colonna `invocation_id` (o `timestamp`) nel `write_run` — 5 righe di modifica. Alternativamente, in pandas: `df["run_global"] = df.groupby(["n","u","n_bits","hash_k","degree","set_size","overlap"]).cumcount()`.

### 5. Colonna `backend` per confronti tra protocolli
Se in futuro unifichi in un solo CSV i risultati di `bench_e2e` (Alg. 1, `backend=bfv+pasta3`) e `bench_e2e_alt` (Alg. 2, `backend=bfv+pasta3-alt`), **filtra sempre** su `backend` prima di aggregare. Le due tabelle hanno **header diversi** (Alg. 1 ha 24 colonne, Alg. 2 ne ha 16), quindi in realtà vanno tenute separate — ma se proprio vuoi unirle, fai un `concat` con `pd.concat([df1, df2], join="outer")` e riempi i NaN.

### 6. Parametri della run: dove guardarli
I parametri sono nelle colonne 2–8 (`n`, `u`, `n_bits`, `hash_k`, `degree`, `set_size`, `overlap`). Sono **ridondanti** in ogni riga: anche se il file cresce, non serve un dizionario esterno per ricostruire i parametri di ogni run. Questo è voluto per rendere il CSV auto-contenuto.

---

## Riepilogo per la tesi

Puoi descrivere il file così:

> Il benchmark produce un file CSV auto-contenuto con una riga per run. Le prime 8 colonne (`run`, `n`, `u`, `n_bits`, `hash_k`, `degree`, `set_size`, `overlap`) riportano i parametri della run; la nona (`backend`) identifica il protocollo (`bfv+pasta3-alt` per Algorithm 2); le ultime 7 riportano misure e output: il tempo totale in secondi (`total_s`), l'indice σ scelto dal TTP (`sigma`), le cardinalità attesa e trovata dell'intersezione (`expected_intersection_size`, `found_intersection_size`), il numero di falsi positivi e falsi negativi (`false_positives`, `false_negatives`) e un flag booleano `match` che vale `true` se e solo se FP = FN = 0. La correttezza del protocollo è verificabile direttamente dal CSV: tutte le righe devono avere `match=true`, `false_positives=0`, `false_negatives=0`.

Se ti serve la versione con timings per-fase (FHE setup, OT, transcipher, AND, CMUX separati) per fare grafici a barre impilate, dimmelo e ti scrivo il patch a `psi.rs` + il `write_run` aggiornato.

run sicura:
for %n in (2 3 4 8 16) do cargo run --release --example bench_e2e -- --n %n --u 4 --n-bits 128 --hash-k 3 --degree 0 --runs 3 --set-size 10 --out f1_secure.csv --append